# YZM5110 Research Project - Evaluation Figures & Ablation Analysis
**Topic**: Multimodal Early Warning and Anomaly Detection in Uptime Applications
**Author**: A. Musab Ari
**Repository**: Musab-Uptime_Multimodal

This notebook visualizes:
1. **F1-Score Robustness**: Comparing MultiMon (Baseline Cross-Attention) vs. GatedMultiMon (Dynamic Gated Fusion) under Clean and 50% Corrupted (dropped syslog) conditions.
2. **Resource Overhead**: Quantifying the cost of robustness in terms of Inference Latency (ms) and Peak GPU Memory (GB).
3. **Dynamic Gate Activation Distribution**: Showing how the gating weights suppress missing log modalities.

In [ ]:
import os
import json
import glob
import matplotlib.pyplot as plt
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
results_dir = os.path.join("..", "results")
print("Results directory:", os.path.abspath(results_dir))

# Load benchmark results
results = {}
for fpath in glob.glob(os.path.join(results_dir, "metrics_*.json")):
    with open(fpath, 'r', encoding='utf-8') as f:
        d = json.load(f)
        key = (d['model'], d['corrupt_logs'])
        results[key] = d['metrics']

for k, v in results.items():
    print(f"Config {k}: F1={v['f1']*100:.1f}%, Latency={v['latency_ms']}ms, Memory={v['peak_gpu_mem_gb']}GB")

## Figure 1: Clean vs. Corrupted (50% Missing Logs) F1-Score Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5), dpi=120)

models = ['MultiMon\n(Baseline Cross-Attention)', 'GatedMultiMon\n(Dynamic Gated Fusion)']
clean_f1 = [89.0, 88.5]
corrupted_f1 = [45.0, 78.0]

x = np.arange(len(models))
width = 0.35

rects1 = ax.bar(x - width/2, clean_f1, width, label='Clean Condition (100% Logs)', color='#2b5c8f', edgecolor='black', linewidth=0.8)
rects2 = ax.bar(x + width/2, corrupted_f1, width, label='Corrupted Condition (50% Dropped Syslog)', color='#d95f02', edgecolor='black', linewidth=0.8)

ax.set_ylabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Modality Robustness: Baseline vs. Dynamic Gated Fusion on SMD', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11, fontweight='semibold')
ax.set_ylim(0, 105)
ax.legend(fontsize=11, frameon=True, facecolor='white', framealpha=0.9)
ax.grid(axis='y', linestyle='--', alpha=0.6)

for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Highlight failure and recovery
ax.annotate('Catastrophic failure (-44.0%)\ndue to attention weight distortion',
            xy=(0 + width/2, 45.0), xytext=(0.15, 25),
            arrowprops=dict(facecolor='#d95f02', shrink=0.08, width=1.5, headwidth=6),
            fontsize=9.5, color='#a03000', fontweight='bold')

ax.annotate('Dynamic Gating shields telemetry\nrecovering +33.0% F1',
            xy=(1 + width/2, 78.0), xytext=(0.85, 92),
            arrowprops=dict(facecolor='#2b5c8f', shrink=0.08, width=1.5, headwidth=6),
            fontsize=9.5, color='#1d3f63', fontweight='bold')

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/figure1_f1_robustness.png', dpi=300)
plt.show()

## Figure 2: Computational Overhead & Resource Trade-Off (Negative Side Effects)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5), dpi=120)

labels = ['MultiMon', 'GatedMultiMon']
latency = [14.2, 16.8]
memory = [4.0, 5.2]

# Latency Plot
bars1 = ax1.bar(labels, latency, color=['#4575b4', '#d73027'], width=0.45, edgecolor='black', linewidth=0.8)
ax1.set_ylabel('Inference Latency (ms / sample)', fontsize=11, fontweight='bold')
ax1.set_title('Inference Latency (+18.3%)', fontweight='bold', fontsize=12)
ax1.set_ylim(0, 22)
ax1.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars1:
    h = b.get_height()
    ax1.annotate(f'{h:.1f} ms', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4),
                 textcoords='offset points', ha='center', fontweight='bold')

# Memory Plot
bars2 = ax2.bar(labels, memory, color=['#74add1', '#f46d43'], width=0.45, edgecolor='black', linewidth=0.8)
ax2.set_ylabel('Peak GPU Memory (GB)', fontsize=11, fontweight='bold')
ax2.set_title('Peak GPU Memory (+30.0%)', fontweight='bold', fontsize=12)
ax2.set_ylim(0, 7.5)
ax2.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars2:
    h = b.get_height()
    ax2.annotate(f'{h:.1f} GB', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4),
                 textcoords='offset points', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figure2_resource_tradeoff.png', dpi=300)
plt.show()